In [3]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera.pandas as pa  # new import
from pandera import Column, DataFrameSchema, Check
from common.loaders import load_valid_country_ids


# ======================
# Definición de esquema
# ======================

languages_schema = DataFrameSchema({
    "id_country": Column(
        str,
        [
            Check.str_length(3, 3),
            Check.isin(load_valid_country_ids())
        ],
        nullable=False
    ),
    "id_language": Column(str, nullable=False),
    "language_name": Column(str, nullable=False),
})


# ======================
# Tareas
# ======================

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,languages"


@task
def extract_languages():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()


@task
def transform_languages(data):
    df = pd.DataFrame(data)

    # Convertir diccionario a lista de (id_language, language_name)
    df_lang = (
        df.assign(
            language_list=df["languages"].apply(lambda x: list(x.items()) if isinstance(x, dict) else [])
        )
        .explode("language_list")
        .dropna(subset=["language_list"])
    )

    # Separar clave y valor
    df_lang[["id_language", "language_name"]] = pd.DataFrame(
        df_lang["language_list"].tolist(), index=df_lang.index
    )

    # Renombrar y limpiar
    df_lang = df_lang.rename(columns={"cca3": "id_country"})[
        ["id_country", "id_language", "language_name"]
    ]

    return df_lang


@task
def validate_languages(df: pd.DataFrame) -> pd.DataFrame:
    return languages_schema.validate(df)


@task
def load_languages(df: pd.DataFrame):
    file_path = Path("../stage/country_languages.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")


# ======================
# Flow principal
# ======================

@flow(name="etl_languages_flow")
def etl_languages():
    data = extract_languages()
    df = transform_languages(data)
    df = validate_languages(df)
    load_languages(df)


if __name__ == "__main__":
    etl_languages()


C:\Users\Sebastian\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandera\_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


04:20:46.734 | INFO    | Flow run 'peridot-cassowary' - Beginning flow run 'peridot-cassowary' for flow 'etl_languages_flow'

04:20:47.877 | INFO    | Task run 'extract_languages-734' - Finished in state Completed()

04:20:48.130 | INFO    | Task run 'transform_languages-b3c' - Finished in state Completed()

04:20:48.428 | INFO    | Task run 'validate_languages-ab2' - Finished in state Completed()

Saved 415 rows to ..\stage\country_languages.csv


04:20:48.657 | INFO    | Task run 'load_languages-518' - Finished in state Completed()

04:20:48.707 | INFO    | Flow run 'peridot-cassowary' - Finished in state Completed()